# SMS Spam Detection — Balanced 10,000-Message Dataset

**Algorithm:** TF-IDF (unigrams + bigrams) + Linear Support Vector Machine (SVM).  
**Important:** 5,000 spam + 5,000 ham, but many examples are **synthetic** (template-generated). The test set contains only held-out original SMS.  
This notebook works locally in **VS Code**, with no Google Colab upload step. See `README.md` for setup instructions.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, accuracy_score, precision_score, recall_score, f1_score
from train import DATA_PATH, load_and_split
from predict import predict_sms

if not Path('train.py').exists():
    raise RuntimeError('Please open the entire SMS_Spam_SVM_10k folder in VS Code before running this notebook.')
print('Dataset:', DATA_PATH)

## 1. Load and validate CSV

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
print('Dataset shape:', df.shape)
print('Labels:', df['v1'].value_counts().to_dict())
print('\nProvenance:')
print(pd.crosstab(df['v1'], df['source']))
print('Missing values:', df.isna().sum().to_dict())
print('Exact duplicate texts:', df['v2'].duplicated().sum())
display(df[['v1', 'v2', 'source']].head(8))

## 2. Exploratory data analysis

In [ ]:
df['word_count'] = df['v2'].str.split().str.len()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df['v1'].value_counts()
axes[0].bar(['ham', 'spam'], [counts['ham'], counts['spam']])
axes[0].set_title('Exactly 5,000 samples in each class')
axes[0].set_ylabel('Count')
for label in ['ham', 'spam']:
    axes[1].hist(df.loc[df['v1'] == label, 'word_count'], bins=35,
                 alpha=0.55, label=label)
axes[1].legend()
axes[1].set_xlabel('Words per SMS')
axes[1].set_title('Message length distribution')
plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for axis, label in zip(axes, ['spam','ham']):
    text = ' '.join(df.loc[df['v1'] == label, 'v2'])
    wc = WordCloud(background_color='white', width=700, height=350, max_words=130).generate(text)
    axis.imshow(wc)
    axis.set_title(label.upper())
    axis.axis('off')
plt.tight_layout()
plt.show()

## Data split — evaluate on original SMS only

Original real SMS messages are stratified into an 80% training group and a 20% test group. **Every synthetic record goes into training only**, never into the final test. For model selection, grouped cross-validation keeps variations from the same synthetic template in one fold, and TF-IDF is fitted separately in every fold.

In [ ]:
train_df, test_df = load_and_split()
print('Train rows:', len(train_df), ' | class counts:', train_df['v1'].value_counts().to_dict())
print('Held-out ORIGINAL test rows:', len(test_df), ' | counts:', test_df['v1'].value_counts().to_dict())
assert set(test_df['source']) == {'original'}

## 3. SVM training with TF-IDF and grouped 5-fold CV

In [ ]:
from train import train_model
model = train_model()  # Uses a grouped 5-fold CV to select SVM C, saves the model.

## 4. Evaluate on unseen original messages

In [ ]:
y_true = test_df['v1']
y_pred = model.predict(test_df['v2'])
print('Accuracy:      ', round(accuracy_score(y_true, y_pred), 4))
print('Spam precision:', round(precision_score(y_true, y_pred, pos_label='spam'), 4))
print('Spam recall:   ', round(recall_score(y_true, y_pred, pos_label='spam'), 4))
print('Spam F1:       ', round(f1_score(y_true, y_pred, pos_label='spam'), 4))
print('\nClassification report:')
print(classification_report(y_true, y_pred))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=['ham', 'spam'], cmap='Blues')
plt.title('Held-out genuine SMS only')
plt.show()

In [ ]:
mistakes = test_df.loc[y_pred != y_true, ['v1', 'v2']].copy()
mistakes['predicted'] = y_pred[y_pred != y_true]
print('Incorrect predictions:', len(mistakes))
display(mistakes.head(15))

## 5. Predict new SMS messages

In [ ]:
examples = [
    'Congratulations! You have won a free phone. Claim your prize now!',
    'Hi, are we meeting tomorrow at 3 PM?',
    'Your package is held. Pay the delivery fee at https://parcel.example.com.',
    'I will send the assignment notes after class.'
]
for message in examples:
    print(f'{predict_sms(message, model):4s} | {message}')

**Interpretation:** A high score on the historical held-out original dataset does not guarantee equal performance on new spam. Synthetic text is useful for augmentation but is not independent real evidence. For research evaluation, collect an external newer labeled test set.